# Primer3 Thermodynamics

Score DNA oligos for the thermodynamic properties that decide whether a primer works: melting temperature (Tm), hairpin and dimer stability (ΔG), GC content, and a 3' GC-clamp. Pair a forward primer with its reverse as a `partner` to check the pair for cross-dimerization.

This wraps [primer3-py](https://github.com/libnano/primer3-py), the Cython binding to [Primer3](https://primer3.org/) ([Untergasser et al., 2012](https://doi.org/10.1093/nar/gks596)).

This example is oriented toward **qPCR** primer design, but the metrics apply to PCR and sequencing primers generally.

In [ ]:
from proto_tools.tools.sequence_scoring.primer3 import (
    Primer3ThermodynamicsConfig,
    Primer3ThermodynamicsInput,
    run_primer3_thermodynamics,
)
from proto_tools.utils.notebook_docs import display_api_reference

## API reference

In [ ]:
display_api_reference("primer3-thermodynamics", "input", "run_primer3_thermodynamics")

In [ ]:
display_api_reference("primer3-thermodynamics", "config", "run_primer3_thermodynamics")

In [ ]:
display_api_reference("primer3-thermodynamics", "output", "run_primer3_thermodynamics")

## Basic usage: score a qPCR primer pair

A GAPDH-style forward/reverse pair. We pass the reverse primer as the forward's `partner`, so the heterodimer ΔG between the two is computed.

In [ ]:
fwd = "ACCCACTCCTCCACCTTTGA"
rev = "CTGTTGCTGTAGCCAAATTCGT"

result = run_primer3_thermodynamics(
    Primer3ThermodynamicsInput(oligos=[{"sequence": fwd, "partner": rev}]),
    Primer3ThermodynamicsConfig(),
)

r = result.results[0]
print(f"Tm:             {r.tm:.1f} °C   (qPCR target 58-62 °C)")
print(f"GC content:     {r.gc_content:.0%}      (target 40-60%)")
print(f"GC clamp:       {r.gc_clamp}")
print(f"Hairpin ΔG:     {r.hairpin_dg:.2f} kcal/mol   (want > -2)")
print(f"Homodimer ΔG:   {r.homodimer_dg:.2f} kcal/mol   (want > -6)")
print(f"Heterodimer ΔG: {r.heterodimer_dg:.2f} kcal/mol   (fwd x rev; want > -6)")

## Advanced usage: batch scoring under qPCR conditions

`primer3-py`'s defaults reproduce Primer3 directly; they are not a qPCR preset. Set the ionic/oligo conditions to match your master mix (here: higher Mg2+, dNTP, and oligo concentration). Multiple oligos are scored in one call and returned in input order.

In [ ]:
qpcr_config = Primer3ThermodynamicsConfig(
    dv_conc=3.0,      # Mg2+ (mM)
    dntp_conc=0.8,    # dNTP (mM)
    dna_conc=250.0,   # oligo (nM)
    temp_c=60.0,      # evaluate hairpin/dimer ΔG near the annealing temperature
)

candidates = [
    "ACCCACTCCTCCACCTTTGA",
    "GTGGTGAAGCAGGCATCTGA",
    "CCCCCATCCGCTAGGGGGG",   # GC-rich: expect a strong hairpin
]

batch = run_primer3_thermodynamics(
    Primer3ThermodynamicsInput(oligos=candidates),
    qpcr_config,
)

print(f"{'oligo':22} {'Tm':>6} {'GC':>5} {'clamp':>6} {'hairpin':>8} {'homodimer':>10}")
for seq, res in zip(candidates, batch.results):
    print(f"{seq:22} {res.tm:6.1f} {res.gc_content:5.0%} {str(res.gc_clamp):>6} {res.hairpin_dg:8.2f} {res.homodimer_dg:10.2f}")

## Export results

In [ ]:
from pathlib import Path

out_dir = Path("./primer3_results")
out_dir.mkdir(exist_ok=True)
batch.export(name="primers", export_path=str(out_dir), file_format="csv")
batch.export(name="primers", export_path=str(out_dir), file_format="json")

print("Wrote:", sorted(p.name for p in out_dir.iterdir()))